1. Create a simple neural network using Keras to classify handwritten digits from the MNIST dataset, then experiment by changing the number of layers and neurons in your model to see how accuracy changes.<br><br><em><strong>Hint:</strong> Start with one hidden layer, then try adding another, and vary the number of neurons (e.g., 32, 64, 128).</em>


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

model = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(x_train, y_train, epochs=10, validation_split=0.2)

loss, accuracy = model.evaluate(x_test, y_test)

print("Test Accuracy:", accuracy)

2. Train your MNIST model using different batch sizes (e.g., 16, 32, 64) and epochs (e.g., 5, 10, 20), and record how the training time and validation accuracy change for each combination.

In [ ]:
import tensorflow as tf
import time

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0

batch_sizes = [16, 32, 64]
epochs_list = [5, 10, 20]

for batch in batch_sizes:
    for epochs in epochs_list:

        model = tf.keras.Sequential([
            tf.keras.layers.Flatten(input_shape=(28,28)),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(10, activation="softmax")
        ])

        model.compile(
            optimizer="adam",
            loss="sparse_categorical_crossentropy",
            metrics=["accuracy"]
        )

        start = time.time()

        history = model.fit(
            x_train,
            y_train,
            epochs=epochs,
            batch_size=batch,
            validation_split=0.2,
            verbose=0
        )

        end = time.time()

        print(
            "Batch:",
            batch,
            "Epochs:",
            epochs,
            "Time:",
            round(end-start,2),
            "Validation Accuracy:",
            history.history["val_accuracy"][-1]
        )

3. Modify your Keras model to use three different optimizers (SGD, Adam, RMSprop) and compare the results by plotting training and validation loss for each optimizer.

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0

optimizers = {
    "SGD": tf.keras.optimizers.SGD(),
    "Adam": tf.keras.optimizers.Adam(),
    "RMSprop": tf.keras.optimizers.RMSprop()
}

for name, opt in optimizers.items():

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28,28)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer=opt,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history = model.fit(
        x_train,
        y_train,
        epochs=10,
        validation_split=0.2,
        verbose=0
    )

    plt.figure()
    plt.plot(history.history["loss"], label="Training Loss")
    plt.plot(history.history["val_loss"], label="Validation Loss")
    plt.title(name)
    plt.legend()
    plt.show()

4. Use KerasTuner to automatically search for the best number of neurons in the first hidden layer for your MNIST classifier. Show the code you used and report the best configuration found.<br><br><em><strong>Hint:</strong> Use Hyperband or RandomSearch from keras_tuner and set a reasonable search space for neurons (e.g., 32 to 256).</em>

In [ ]:
import tensorflow as tf
import keras_tuner as kt

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0

def build_model(hp):

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28,28)),
        tf.keras.layers.Dense(
            units=hp.Int("neurons",32,256,step=32),
            activation="relu"
        ),
        tf.keras.layers.Dense(10, activation="softmax")
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

tuner = kt.Hyperband(
    build_model,
    objective="val_accuracy",
    max_epochs=10,
    directory="my_dir",
    project_name="mnist_tuning"
)

tuner.search(
    x_train,
    y_train,
    epochs=10,
    validation_split=0.2
)

best_model = tuner.get_best_models(1)[0]

best_hp = tuner.get_best_hyperparameters(1)[0]

print("Best Number of Neurons:", best_hp.get("neurons"))

5. Experiment with different weight initialization methods (e.g., 'he_normal', 'glorot_uniform', 'random_normal') in your Keras model and compare their impact on the model's training speed and accuracy.

In [ ]:
import tensorflow as tf

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

x_train = x_train / 255.0
x_test = x_test / 255.0

initializers = [
    "he_normal",
    "glorot_uniform",
    "random_normal"
]

for init in initializers:

    model = tf.keras.Sequential([
        tf.keras.layers.Flatten(input_shape=(28,28)),
        tf.keras.layers.Dense(
            64,
            activation="relu",
            kernel_initializer=init
        ),
        tf.keras.layers.Dense(
            10,
            activation="softmax",
            kernel_initializer=init
        )
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    model.fit(
        x_train,
        y_train,
        epochs=10,
        validation_split=0.2,
        verbose=0
    )

    loss, accuracy = model.evaluate(x_test, y_test, verbose=0)

    print(init, "Accuracy:", accuracy)